# NB_02 — Electroplated Bi Process Window v3

**Engineering question**

> What electroplated-bismuth thickness and microstructure process window preserves Gaussian-like spectral response while increasing x-ray stopping power?

Version 3 makes this notebook primarily an **orchestration layer**.

Reusable process-window logic now lives in:

```text
tools/
    optimization/
        process_window_builder.py
```

The notebook loads Engineering Objects and completed source records, calls the reusable builder, inspects the results, and writes/export the engineering artifacts.

The builder remains conservative: numerical tolerances are left unresolved where the repository does not yet contain replicated process-to-response measurements.


## Workflow

```text
Engineering Objects
        +
SOURCE records
        ↓
process_window_builder.py
        ↓
process variables
quantitative evidence
operating points
coupled variables
candidate process window
validation matrix
status
        ↓
NB_02 display + export
```


## 1. Locate repository and import the reusable process-window engine

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import shutil
import subprocess
import sys
import zipfile

import pandas as pd
import yaml

REPOSITORY_URL = "https://github.com/thinkthoughts/sensors-becker.git"
REPO_ROOT_OVERRIDE: str | Path | None = None

SOURCE_FILES = [
    "SOURCE_00_becker_transition_models.yaml",
    "SOURCE_01_bismuth_microstructure.yaml",
    "SOURCE_02_eliminating_nongaussian_spectral_response.yaml",
]

NOTEBOOK_ID = "NB_02_ELECTROPLATED_BI_PROCESS_WINDOW"
PROCESS_WINDOW_ID = "PROCESS_WINDOW_01"


def find_repo_root() -> Path:
    candidates = []

    if REPO_ROOT_OVERRIDE is not None:
        candidates.append(Path(REPO_ROOT_OVERRIDE).expanduser().resolve())

    start = Path.cwd().resolve()
    candidates.extend([start, *start.parents])

    candidates.extend([
        Path("/content/sensors-becker"),
        Path("/home/dan/sensors-becker"),
        Path.home() / "sensors-becker",
    ])

    for candidate in candidates:
        if (
            candidate.is_dir()
            and (candidate / "engineering_navigator").is_dir()
        ):
            return candidate

    if Path("/content").exists():
        target = Path("/content/sensors-becker")

        if target.exists():
            if (target / "engineering_navigator").is_dir():
                return target
            raise FileExistsError(
                f"{target} exists but does not look like sensors-becker."
            )

        subprocess.run(
            ["git", "clone", REPOSITORY_URL, str(target)],
            check=True,
        )

        if (target / "engineering_navigator").is_dir():
            return target

    raise FileNotFoundError(
        "Could not locate sensors-becker repository. "
        "Set REPO_ROOT_OVERRIDE explicitly."
    )


ROOT = find_repo_root()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OBJ_DIR = ROOT / "engineering_navigator" / "engineering_objects"
SOURCE_DIR = (
    ROOT
    / "engineering_navigator"
    / "absorber_manufacturing"
    / "source_records"
)
OUTPUT_DIR = (
    ROOT
    / "outputs"
    / "engineering_questions"
    / "absorber_manufacturing"
    / PROCESS_WINDOW_ID
)
EXPORT_DIR = ROOT / "exports" / PROCESS_WINDOW_ID
EXPORT_ZIP = ROOT / "exports" / f"{PROCESS_WINDOW_ID}_export.zip"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)

from tools.optimization.process_window_builder import (
    DEFAULT_ELECTROPLATED_BI_CONFIG,
    build_process_window,
)

print(f"Repository : {ROOT}")
print("Imported   : tools.optimization.process_window_builder")


## 2. Load Engineering Objects and completed source records

In [ ]:
def load_yaml(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(path)

    data = yaml.safe_load(path.read_text(encoding="utf-8"))

    if not isinstance(data, dict):
        raise TypeError(f"{path}: expected one top-level YAML mapping")

    return data


engineering_objects = {
    object_id: load_yaml(OBJ_DIR / f"{object_id}.yaml")
    for object_id in ("absorber", "electroplating", "tes")
}

records = {}

for filename in SOURCE_FILES:
    record = load_yaml(SOURCE_DIR / filename)
    source_id = record.get("source_id")

    if not source_id:
        raise KeyError(f"{filename}: missing source_id")

    if source_id in records:
        raise ValueError(f"Duplicate source_id: {source_id}")

    records[source_id] = record


incomplete = {
    source_id: record.get("extraction_status")
    for source_id, record in records.items()
    if not str(record.get("extraction_status", "")).startswith("complete")
}

if incomplete:
    raise ValueError(
        "Incomplete source records: " + json.dumps(incomplete, indent=2)
    )

print("Engineering Objects:", ", ".join(engineering_objects))
print("Source-record validation: PASS")
print("Sources:", ", ".join(sorted(records)))


## 3. Build the process-window package

In [ ]:
result = build_process_window(
    engineering_objects=engineering_objects,
    source_records=records,
    config=DEFAULT_ELECTROPLATED_BI_CONFIG,
)

print("Process-window build: PASS")
print(f"Process variables        : {len(result.process_variables)}")
print(f"Quantitative evidence    : {len(result.quantitative_evidence)}")
print(f"Operating points         : {len(result.operating_points)}")
print(f"Coupled variables        : {len(result.coupled_variables)}")
print(f"Window dimensions        : {len(result.candidate_process_window)}")
print(f"Validation experiments   : {len(result.validation_matrix)}")


## 4. Process variables

In [ ]:
result.process_variables


## 5. Quantitative evidence

In [ ]:
result.quantitative_evidence


## 6. Source-supported operating points

In [ ]:
if result.operating_points.empty:
    print("No source-supported process operating points recorded.")
else:
    result.operating_points


## 7. Coupled Engineering Object variables

In [ ]:
result.coupled_variables


## 8. Candidate process-window dimensions

A blank `candidate_range` is intentional. It means the repository does not yet contain enough replicated evidence to justify a numerical manufacturing tolerance.


In [ ]:
result.candidate_process_window


## 9. Validation matrix

In [ ]:
result.validation_matrix


## 10. Process-window status

In [ ]:
process_window_status = {
    "notebook_id": NOTEBOOK_ID,
    **result.status,
}

process_window_status


## 11. Engineering interpretation

The reusable builder distinguishes between:

- **observed operating points** — values already reported in the source records or Engineering Objects;
- **candidate process-window dimensions** — variables that should eventually receive tolerances;
- **validation experiments** — measurements required before a numerical range can be justified.

The notebook therefore does not convert a single reported electroplating recipe into a manufactured tolerance range.


## 12. Write outputs

In [ ]:
process_variables_csv = OUTPUT_DIR / "process_variables.csv"
quantitative_evidence_csv = OUTPUT_DIR / "quantitative_evidence.csv"
operating_points_csv = OUTPUT_DIR / "operating_points.csv"
coupled_variables_csv = OUTPUT_DIR / "coupled_variables.csv"
window_csv = OUTPUT_DIR / "candidate_process_window.csv"
validation_csv = OUTPUT_DIR / "validation_matrix.csv"
status_json = OUTPUT_DIR / "process_window_status.json"

result.process_variables.to_csv(process_variables_csv, index=False)
result.quantitative_evidence.to_csv(quantitative_evidence_csv, index=False)
result.operating_points.to_csv(operating_points_csv, index=False)
result.coupled_variables.to_csv(coupled_variables_csv, index=False)
result.candidate_process_window.to_csv(window_csv, index=False)
result.validation_matrix.to_csv(validation_csv, index=False)

status_json.write_text(
    json.dumps(
        process_window_status,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

written_files = {
    "process_variables": process_variables_csv,
    "quantitative_evidence": quantitative_evidence_csv,
    "operating_points": operating_points_csv,
    "coupled_variables": coupled_variables_csv,
    "candidate_process_window": window_csv,
    "validation_matrix": validation_csv,
    "process_window_status": status_json,
}

for name, path in written_files.items():
    print(f"{name:26} {path.relative_to(ROOT)}")


## 13. Build and download export ZIP

In [ ]:
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for path in written_files.values():
    shutil.copy2(path, EXPORT_DIR / path.name)

if EXPORT_ZIP.exists():
    EXPORT_ZIP.unlink()

with zipfile.ZipFile(
    EXPORT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(EXPORT_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

print(f"Export package: {EXPORT_ZIP}")
print(f"Size: {EXPORT_ZIP.stat().st_size:,} bytes")

try:
    from google.colab import files
    files.download(str(EXPORT_ZIP))
except ImportError:
    print("Automatic download is available only in Google Colab.")


## 14. Handoff

Version 3 establishes a reusable process-window architecture:

```text
Engineering Objects
        +
Source Records
        ↓
tools.optimization.process_window_builder
        ↓
NB_02 orchestration
        ↓
Process-window evidence + validation plan
```

The next substantive engineering advance requires additional source or experimental records containing **replicated process-to-response measurements**. Once those exist, the reusable builder can be extended to estimate evidence-supported numerical ranges rather than leaving them unresolved.

*Admissible generalizations trail leading specifications.*
